# E11-02 · Build the SGLang 0.5.21 offline wheelhouse → Kaggle dataset (runs ON Kaggle)

**Settings:** Accelerator **None** (CPU, no GPU quota) · Internet **ON** · Add-ons → Secrets → attach `KAGGLE_USERNAME`
and `KAGGLE_KEY` · start with **Save Version → Save & Run All** from the editor (secrets are not readable in API-started runs).

**What:** every wheel of the SGLang 0.5.21 stack for this exact Kaggle environment (CPython 3.12, glibc 2.35, x86_64):
torch 2.13.0 (CUDA 13 runtime wheels), flashinfer-python 0.6.18 + flashinfer-jit-cache 0.6.18+cu130 (precompiled kernels,
no offline JIT stall), transformers 5.12.1, sglang-kernel 0.4.7 - 220 wheels, about 5 GB.

**Why a pinned lock instead of `pip download sglang`:** sglang 0.5.19-0.5.21 all pin `cuda-tile==1.6.0rc5`, which exists
on PyPI only as a source tarball, so a binary-only download fails. The lock was resolved off-Kaggle with
`uv pip compile --override cuda-tile==1.6.0` for Linux / CPython 3.12 (cuda-tile 1.6.0 has a cp312 wheel). It is
installed with **`--no-deps`**, so pip never re-checks that one pin.

**CUDA compiler wheels:** `nvidia-cuda-nvcc` / `-crt` / `nvidia-nvvm` / `nvidia-nvjitlink` are pinned to **13.0.88**, the set
torch 2.13's own `cuda-toolkit 13.0.3` defines (cudart 13.0.96). Kaggle's L4 driver supports CUDA 13.0. Without the pin, an
unpinned dependency (humming-kernels) pulls nvcc 13.4.92.

**Version 2 (2026-10-15):** run 1 published the dataset, but Kaggle drops `+` from file names
(`flashinfer_jit_cache-0.6.18+cu130-...whl` became `...-0.6.18cu130-...whl`), so a plain `pip install --find-links` cannot
match the lock. The dataset now ships **`install_wheelhouse.py`**; this notebook tests it on a Kaggle-style copy (`+`
removed) and then publishes a new version of the same dataset.

**Install in the GPU notebook (a separate venv, the image's own torch stays untouched):**

    python /kaggle/input/<this dataset>/install_wheelhouse.py /tmp/sglang-venv

It restores the names (checking their sha256), creates the venv, installs `requirements.lock` with `--no-deps`, adds
`lib/libcudart.so` + `lib64` under the pip CUDA runtime, and prints `VENV_PYTHON`, `CUDA_HOME` and `PATH`. Start SGLang
with them and a fresh `SGLANG_CACHE_DIR` / `CUDA_CACHE_PATH`.
- Why a venv rather than `--target` + `PYTHONPATH`: `.pth` files run only from a site directory. `nvidia-cutlass-dsl` is
  *only* a `.pth` file.
- Console scripts such as `ninja` must be on `PATH`.


In [ ]:
import datetime, hashlib, json, os, re, shutil, subprocess, sys, time
from pathlib import Path

SGLANG_VERSION = "0.5.21"
PUBLIC = True
DATASET_SLUG = "sglang-0-5-21-cu13-wheelhouse"
DATASET_TITLE = "SGLang 0.5.21 cu13 wheelhouse (py3.12)"
VERSION_NOTES = ("v2: install_wheelhouse.py restores the '+' Kaggle drops from file names "
                 "(flashinfer-jit-cache 0.6.18+cu130) and installs into a venv; README updated")
WH = Path("/kaggle/working/sglang-0.5.21-wheelhouse")      # this is what gets uploaded (~5 GB)
VENV = Path("/tmp/sglang-venv")                              # offline install test (venv), deleted before upload
VPY = VENV / "bin" / "python"
FLASHINFER_INDEX = "https://flashinfer.ai/whl/cu130/"
LOCK = """aiohappyeyeballs==2.7.1
aiohttp==3.14.5
aiosignal==1.4.0
airportsdata==20260905
annotated-doc==0.0.5
annotated-types==0.8.0
anthropic==1.14.0
anyio==4.15.1
apache-tvm-ffi==0.1.11
asttokens==3.0.2
attrs==26.1.0
bleach==6.4.0
blobfile==3.0.0
build==1.6.1
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.2
click==8.5.0
cloudpickle==3.1.2
compressed-tensors==0.18.0
cryptography==50.0.2
cuda-bindings==13.4.4
cuda-core==1.2.1
cuda-pathfinder==1.8.3
cuda-python==13.4.4
cuda-tile==1.6.0
cuda-toolkit==13.0.3.0
cython==3.3.0
datasets==5.1.0
dill==0.4.1
diskcache==5.6.3
distro==1.9.0
docstring-parser==0.18.0
easydict==1.13
einops==0.8.2
executing==2.3.0
fastapi==0.143.1
fastjsonschema==2.22.2
filelock==4.2.0
flash-attn-4==4.0.0b35
flashinfer-jit-cache==0.6.18+cu130
flashinfer-python==0.6.18
frozenlist==1.8.0
fsspec==2026.7.0
gguf==0.19.0
grpcio==1.84.0
grpcio-health-checking==1.81.1
grpcio-reflection==1.81.1
h11==0.16.0
hf-xet==1.7.0
httpcore==1.0.9
httpcore2==2.13.1
httpx==0.28.1
httpx2==2.13.1
huggingface-hub==1.33.0
humming-kernels==0.1.12
idna==3.20
interegular==0.3.3
ipython==9.17.1
ipython-pygments-lexers==1.1.1
jedi==0.20.1
jinja2==3.1.6
jiter==0.17.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-core==5.9.1
jupytext==1.19.6
kaggle==2.2.4
kagglesdk==0.1.37
kernels==0.14.1
kernels-data==0.16.2
lark==1.3.1
llguidance==1.9.1
llvmlite==0.47.0
loguru==0.7.3
lxml==6.1.3
markdown-it-py==4.2.0
markupsafe==3.0.4
matplotlib-inline==0.2.2
mdit-py-plugins==0.6.1
mdurl==0.1.2
mistral-common==1.12.0
ml-dtypes==0.6.0
modelscope==1.40.2
modelscope-hub==0.4.6
mpmath==1.3.0
msgspec==0.22.0
multidict==7.1.0
multiprocess==0.70.19
nbformat==5.11.1
nccl4py==0.6.0
nest-asyncio==1.6.0
networkx==3.7
ninja==1.13.2
numba==0.65.1
numpy==2.3.5
nvidia-cublas==13.1.1.3
nvidia-cuda-cccl==13.3.4.3.1
nvidia-cuda-crt==13.0.88
nvidia-cuda-cupti==13.0.85
nvidia-cuda-nvcc==13.0.88
nvidia-cuda-nvdisasm==13.4.92
nvidia-cuda-nvrtc==13.0.88
nvidia-cuda-runtime==13.0.96
nvidia-cudnn-cu13==9.20.0.48
nvidia-cudnn-frontend==1.31.0
nvidia-cufft==12.0.0.61
nvidia-cufile==1.15.1.6
nvidia-curand==10.4.0.35
nvidia-cusolver==12.0.4.66
nvidia-cusparse==12.6.3.3
nvidia-cusparselt-cu13==0.8.1
nvidia-cutlass-dsl==4.6.2
nvidia-cutlass-dsl-libs-base==4.6.2
nvidia-cutlass-dsl-libs-core==4.6.2
nvidia-cutlass-dsl-libs-cu12==4.6.2
nvidia-cutlass-dsl-libs-cu13==4.6.2
nvidia-mathdx==25.6.0
nvidia-ml-py==13.615.71
nvidia-nccl-cu13==2.29.7
nvidia-nvjitlink==13.0.88
nvidia-nvshmem-cu13==3.4.5
nvidia-nvtx==13.0.85
nvidia-nvvm==13.0.88
nvshmem4py-cu13==0.4.0
openai==2.6.1
openai-harmony==0.0.4
opentelemetry-api==1.45.1
orjson==3.13.0
outlines==0.1.11
outlines-core==0.1.26
packaging==26.3
pandas==3.0.6
parso==0.8.7
partial-json-parser==0.2.1.1.post7
pexpect==4.9.0
pillow==12.3.0
platformdirs==4.13.0
prometheus-client==0.26.0
prompt-toolkit==3.0.53
propcache==0.5.4
protobuf==6.33.6
psutil==7.2.2
ptyprocess==0.7.0
pure-eval==0.2.4
py-spy==0.4.2
pyarrow==26.0.0
pybase64==1.5.1
pycountry==26.2.16
pycparser==3.11
pycryptodomex==3.24.1
pydantic==2.14.1
pydantic-core==2.50.1
pydantic-extra-types==2.11.1
pygments==2.21.0
pyproject-hooks==1.3.3
python-dateutil==2.9.0.post0
python-dotenv==1.2.4
python-multipart==0.0.32
python-slugify==9.1.3
pyyaml==6.0.3
pyzmq==27.2.0
quack-kernels==0.6.4
referencing==0.37.0
regex==2026.9.29
requests==2.34.2
rich==15.0.0
rpds-py==2026.9.1
safetensors==0.8.0
scipy==1.18.1
sentencepiece==0.2.1
setproctitle==1.3.8
setuptools==84.0.0
sgl-deep-ep==0.1.2
sgl-deep-gemm==0.2.0
sglang==0.5.21
sglang-kernel==0.4.7
shellingham==1.5.4
six==1.17.0
smg-grpc-proto==0.4.26
smg-grpc-servicer==0.13.6
sniffio==1.3.1
soundfile==0.13.1
stack-data==0.6.3
starlette==1.8.0
sympy==1.14.0
tabulate==0.10.0
text-unidecode==1.3
tiktoken==0.14.0
tilelang==0.1.12
timm==1.0.16
tokenizers==0.22.2
tokenspeed-mla==0.1.8
tokenspeed-triton==3.9.10.post20261008
tomlkit==0.15.1
torch==2.13.0
torch-c-dlpack-ext==0.1.5
torch-memory-saver==0.0.10
torchaudio==2.11.0
torchcodec==0.15.0
torchvision==0.28.0
tqdm==4.70.1
traitlets==5.16.1
transformers==5.12.1
triton==3.7.1
truststore==0.10.5
typer==0.27.3
typing-extensions==4.16.0
typing-inspection==0.4.4
urllib3==2.8.0
uvicorn==0.54.0
uvloop==0.23.0
watchfiles==1.3.0
wcwidth==0.9.2
webencodings==0.6.1
xgrammar==0.2.7
xxhash==4.0.1
yarl==1.25.1
z3-solver==4.15.4.0
zstandard==0.25.0
"""
WH.mkdir(parents=True, exist_ok=True)
(WH / "requirements.lock").write_text(LOCK, encoding="utf-8")
PINS = dict(l.strip().split("==", 1) for l in LOCK.splitlines() if "==" in l)
print("python", sys.version.split()[0], "| pins", len(PINS))
r = subprocess.run(["ldd", "--version"], capture_output=True, text=True)
print("glibc:", (r.stdout or r.stderr).splitlines()[0] if (r.stdout or r.stderr) else "?")
for root in (Path("/kaggle/working"), Path("/tmp")):
    print(f"  free {str(root):16} {shutil.disk_usage(root).free / 1024**3:7.1f} GiB")
assert sys.version_info[:2] == (3, 12), "must run on the same Python as the ARC notebook (3.12)"
assert len(PINS) == 220 and PINS["sglang"] == SGLANG_VERSION and PINS["cuda-tile"] == "1.6.0"
assert PINS["nvidia-cuda-nvcc"] == PINS["nvidia-nvvm"] == PINS["nvidia-nvjitlink"] == "13.0.88"
r = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True)
print(r.stdout.strip())
_pipv = tuple(int(x) for x in r.stdout.split()[1].split(".")[:2])
assert _pipv >= (22, 3), "pip >= 22.3 is needed for pip --python <venv>"

## 1. Download every pinned wheel

In [ ]:
# 1. Download every pinned wheel (binary only, no dependency resolution: the lock is already complete)
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "download", "--no-deps", "--only-binary=:all:", "-r", str(WH / "requirements.lock"),
                    "-d", str(WH), "--extra-index-url", FLASHINFER_INDEX, "--disable-pip-version-check"],
                   capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise SystemExit("pip download failed - paste this output back")
from packaging.utils import parse_wheel_filename, canonicalize_name
have = {}
for w in sorted(WH.glob("*.whl")):
    name, ver, _b, _t = parse_wheel_filename(w.name)
    have.setdefault(canonicalize_name(name), []).append(str(ver))
want = {canonicalize_name(k): v for k, v in PINS.items()}
missing = sorted(k for k in want if k not in have)
dupes = {k: v for k, v in have.items() if len(v) > 1}
wrongv = {k: (have[k], want[k]) for k in want if k in have and want[k] not in have[k]}
size = sum(p.stat().st_size for p in WH.glob("*.whl"))
print(f"wheels: {sum(len(v) for v in have.values())} | {size / 1024**3:.2f} GiB | missing {missing} | duplicates {dupes} | "
      f"version mismatches {wrongv} | {time.time() - t0:.0f}s")
assert not missing and not dupes and not wrongv, "the wheelhouse does not match the lock"

## 2. Offline install test

In [ ]:
# 2. Offline install test through the dataset's own installer, on a copy laid out the way Kaggle stores the files
#    (Kaggle drops '+' from file names: run 1 stored flashinfer_jit_cache-0.6.18+cu130-... as ...-0.6.18cu130-...).
INSTALLER_SRC = r"""
# install_wheelhouse.py - offline install of this SGLang 0.5.21 wheelhouse into a fresh venv (CPython 3.12, Linux x86_64)
#
#     python install_wheelhouse.py [VENV_DIR]          (default /tmp/sglang-venv)
#
# Do NOT run `pip install --find-links <this folder>` directly: Kaggle drops '+' from dataset file names, so
# flashinfer_jit_cache-0.6.18+cu130-cp39-abi3-manylinux_2_28_x86_64.whl is stored as ...-0.6.18cu130-... and pip cannot
# match it to the lock pin. This script:
#   1. links every wheel under its original name (SHA256SUMS keeps the names) into VENV_DIR-wheels, and checks the
#      sha256 of every wheel whose name it had to restore;
#   2. creates the venv without pip (the system pip installs into it with `pip --python`). A venv, not --target +
#      PYTHONPATH, so .pth files run (nvidia-cutlass-dsl is only a .pth) and console scripts (ninja) land in VENV/bin;
#   3. installs requirements.lock with --no-deps (the lock overrides sglang's sdist-only cuda-tile==1.6.0rc5 pin);
#   4. adds lib/libcudart.so and lib64 under the pip CUDA runtime (site-packages/nvidia/cu13): JIT linking uses -lcudart.
# It prints the VENV_PYTHON / CUDA_HOME / PATH to use. On Kaggle also make -lcuda linkable for FlashInfer's JIT (only
# libcuda.so.1 exists there): symlink it as libcuda.so into a directory on LIBRARY_PATH.
import hashlib, os, subprocess, sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
venv = Path(sys.argv[1] if len(sys.argv) > 1 else "/tmp/sglang-venv")
links = Path(os.environ.get("WHEELHOUSE_LINKS", str(venv) + "-wheels"))
links.mkdir(parents=True, exist_ok=True)
restored = []
for line in (HERE / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    digest, name = line.split("  ", 1)
    name = name.strip()
    if not name.endswith(".whl"):
        continue
    src = HERE / name if (HERE / name).exists() else HERE / name.replace("+", "")
    if not src.exists():
        sys.exit(f"wheel missing from this folder: {name}")
    if src.name != name:
        h = hashlib.sha256()
        with open(src, "rb") as f:
            for block in iter(lambda: f.read(1 << 24), b""):
                h.update(block)
        if h.hexdigest() != digest:
            sys.exit(f"sha256 mismatch: {src.name}")
        restored.append(name)
    dst = links / name
    if not dst.exists():
        dst.symlink_to(src)
vpy = venv / "bin" / "python"
if not vpy.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True)
subprocess.run([sys.executable, "-m", "pip", "--python", str(vpy), "install", "--no-index", "--find-links", str(links),
                "--requirement", str(HERE / "requirements.lock"), "--no-deps", "--only-binary", ":all:", "--no-compile",
                "--disable-pip-version-check", "--no-warn-conflicts", "-q"], check=True)
site = Path(subprocess.run([str(vpy), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                           capture_output=True, text=True, check=True).stdout.strip())
cuda = site / "nvidia" / "cu13"
if not (cuda / "lib" / "libcudart.so").exists():
    (cuda / "lib" / "libcudart.so").symlink_to("libcudart.so.13")
if not (cuda / "lib64").exists():
    (cuda / "lib64").symlink_to("lib")
print("restored wheel names:", restored or "none")
print("VENV_PYTHON=" + str(vpy))
print("CUDA_HOME=" + str(cuda))
print("PATH=" + str(venv / "bin") + ":" + str(cuda / "bin") + ":$PATH")
"""
(WH / "install_wheelhouse.py").write_text(INSTALLER_SRC.lstrip("\n"), encoding="utf-8")


def sha256(p, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


t0 = time.time()
WHEEL_SUMS = {w.name: sha256(w) for w in sorted(WH.glob("*.whl"))}
print(f"sha256 of {len(WHEEL_SUMS)} wheels in {time.time() - t0:.0f}s")
SIM = Path("/tmp/kaggle-layout-sim")                          # what a Kaggle mount looks like: '+' removed from names
shutil.rmtree(SIM, ignore_errors=True)
SIM.mkdir(parents=True)
for _n in WHEEL_SUMS:
    (SIM / _n.replace("+", "")).symlink_to(WH / _n)
(SIM / "SHA256SUMS").write_text("".join(f"{h}  {n}\n" for n, h in WHEEL_SUMS.items()), encoding="utf-8")
for _f in ("requirements.lock", "install_wheelhouse.py"):
    shutil.copy(WH / _f, SIM / _f)
print("names changed in the simulated Kaggle layout:", [n for n in WHEEL_SUMS if "+" in n])
shutil.rmtree(VENV, ignore_errors=True)
shutil.rmtree(Path(str(VENV) + "-wheels"), ignore_errors=True)
t0 = time.time()
r = subprocess.run([sys.executable, str(SIM / "install_wheelhouse.py"), str(VENV)], capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise SystemExit("installer failed on the Kaggle-style layout - paste this output back")
assert "flashinfer_jit_cache-0.6.18+cu130" in r.stdout, "the renamed wheel was not restored"
SITE = Path(subprocess.run([str(VPY), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                           capture_output=True, text=True, check=True).stdout.strip())
CUDA_HOME = SITE / "nvidia" / "cu13"
inst = sum(p.stat().st_size for p in VENV.rglob("*") if p.is_file())
print(f"offline install OK in {time.time() - t0:.0f}s | installed size {inst / 1024**3:.1f} GiB | site {SITE}")
print(".pth files:", sorted(p.name for p in SITE.glob("*.pth")))
print("bin:", sorted(p.name for p in (VENV / "bin").iterdir())[:40])

## 3. CPU probes

In [ ]:
# 3. CPU probes (no GPU here): imports, transformers 5.12.1 reads the RedHat INT4 config + tokenizer, SGLang registry
#    knows the target and DFlash2 architectures. Recorded in BUILD_PROBE.json; the GPU notebook re-checks on GPU.
from huggingface_hub import hf_hub_download
SMALL = Path("/tmp/redhat-int4-small"); SMALL.mkdir(exist_ok=True)
for f in ("config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja",
          "processor_config.json"):
    hf_hub_download("RedHatAI/Qwen3.8-27B-INT4", f, revision="91bd022d5b49442a868bc35008f6c21e1860edfa", local_dir=str(SMALL))
DF = Path("/tmp/dflash2-small"); DF.mkdir(exist_ok=True)
hf_hub_download("z-lab/Qwen3.8-27B-DFlash2", "config.json", revision="50307d4c4cde6860d4eee73e2547cd786fe8e8a4", local_dir=str(DF))

PROBE = r"""
import json, sys
out = {}
def step(name, fn):
    try:
        out[name] = fn()
    except Exception as e:
        out[name] = f"ERROR {type(e).__name__}: {str(e)[:300]}"
import importlib
step("torch", lambda: (lambda t: [t.__version__, t.version.cuda])(importlib.import_module("torch")))
step("transformers", lambda: importlib.import_module("transformers").__version__)
step("sglang", lambda: importlib.import_module("sglang").__version__)
step("flashinfer", lambda: importlib.import_module("flashinfer").__version__)
step("sgl_kernel_import", lambda: str(importlib.import_module("sgl_kernel"))[:80])
def cfg():
    from transformers import AutoConfig, AutoTokenizer
    c = AutoConfig.from_pretrained(sys.argv[1])
    t = AutoTokenizer.from_pretrained(sys.argv[1])
    msgs = [{"role": "user", "content": "hi"}]
    txt = t.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    return {"arch": getattr(c, "architectures", None), "model_type": c.model_type, "vocab": len(t), "prompt_tail": txt[-40:]}
step("redhat_config_tokenizer", cfg)
def reg():
    from sglang.srt.models.registry import ModelRegistry
    archs = set(ModelRegistry.get_supported_archs())
    return {a: (a in archs) for a in ("Qwen3_5ForConditionalGeneration", "Qwen3_5ForCausalLM", "DFlash2DraftModel")} | \
           {"n_archs": len(archs), "dflash_like": sorted(a for a in archs if "lash" in a.lower())[:8]}
step("sglang_registry", reg)
step("cutlass_dsl", lambda: [getattr(importlib.import_module("cutlass"), "__version__", "imported"),
                             any("dsl_packages" in p for p in sys.path)])
step("sglang_models_qwen3_5", lambda: str(importlib.import_module("sglang.srt.models.qwen3_5"))[:100])
step("sglang_models_dflash", lambda: str(importlib.import_module("sglang.srt.models.dflash"))[:100])
step("dflash2_config", lambda: (lambda c: {"arch": c.get("architectures"), "layers": c.get("num_hidden_layers")})(
    json.load(open(sys.argv[2]))))
print("PROBE_JSON " + json.dumps(out))
"""
env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_HOME", "CUDA_PATH")}
env.update({"PATH": f"{VENV / 'bin'}:{CUDA_HOME / 'bin'}:{env.get('PATH', '')}", "CUDA_HOME": str(CUDA_HOME),
            "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "SGLANG_CACHE_DIR": "/tmp/sgl-cache"})
r = subprocess.run([str(VPY), "-c", PROBE, str(SMALL), str(DF / "config.json")], env=env, capture_output=True, text=True,
                   timeout=1800)
line = next((l for l in r.stdout.splitlines() if l.startswith("PROBE_JSON ")), None)
PROBE_RESULT = json.loads(line[11:]) if line else {"crash": r.stderr[-2000:]}

# CPU toolchain check of the wheel nvcc 13.0.88 (compiling needs no GPU). In ARC-AGI-3 FlashInfer's JIT never found the
# wheel nvcc; this records what the GPU notebook must set (CUDA_HOME, include / lib paths, a libcudart.so link).
TOOL = {"files": {rel: (CUDA_HOME / rel).exists() for rel in (
    "bin/nvcc", "bin/ptxas", "bin/fatbinary", "nvvm/bin/cicc", "include/cuda_runtime.h", "include/crt/host_config.h",
    "include/cccl", "lib/libcudart.so.13", "lib/libcudart.so", "lib64")}}
def tool(name, cmd):
    try:
        p = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=900)
        TOOL[name] = {"rc": p.returncode, "out": (p.stdout + p.stderr).strip()[-700:]}
    except Exception as e:
        TOOL[name] = {"rc": None, "out": f"{type(e).__name__}: {e}"[:700]}
NVCC = str(CUDA_HOME / "bin" / "nvcc")
TC = Path("/tmp/e11-toolchain"); shutil.rmtree(TC, ignore_errors=True); (TC / "lib").mkdir(parents=True)
(TC / "k.cu").write_text('#include <cuda_runtime.h>\n__global__ void k(float* x) { x[threadIdx.x] *= 2.f; }\n'
                         'extern "C" int launch(float* x) { k<<<1, 32>>>(x); return (int)cudaGetLastError(); }\n')
(TC / "lib" / "libcudart.so").symlink_to(CUDA_HOME / "lib" / "libcudart.so.13")
INC = ["-I", str(CUDA_HOME / "include")]
SM89 = ["-gencode=arch=compute_89,code=sm_89"]
tool("nvcc_version", [NVCC, "--version"])
tool("gcc_version", ["gcc", "--version"])
tool("ninja_version", [str(VENV / "bin" / "ninja"), "--version"])
tool("cubin_sm89", [NVCC, *SM89, "-cubin", *INC, "-o", str(TC / "k.cubin"), str(TC / "k.cu")])
tool("cubin_sm89_no_I", [NVCC, *SM89, "-cubin", "-o", str(TC / "k2.cubin"), str(TC / "k.cu")])
tool("object_sm89", [NVCC, *SM89, "-Xcompiler", "-fPIC", "-c", *INC, "-o", str(TC / "k.o"), str(TC / "k.cu")])
tool("shared_lib_link", [NVCC, *SM89, "-Xcompiler", "-fPIC", "-shared", "-cudart", "shared", *INC,
                         "-L", str(CUDA_HOME / "lib"), "-o", str(TC / "k.so"), str(TC / "k.cu")])
tool("shared_lib_link_with_libcudart_so", [NVCC, *SM89, "-Xcompiler", "-fPIC", "-shared", "-cudart", "shared", *INC,
                                           "-L", str(TC / "lib"), "-o", str(TC / "k3.so"), str(TC / "k.cu")])
PROBE_RESULT["toolchain"] = TOOL
print(json.dumps(PROBE_RESULT, indent=1))
(WH / "BUILD_PROBE.json").write_text(json.dumps(PROBE_RESULT, indent=2), encoding="utf-8")
_bad = [k for k, v in PROBE_RESULT.items() if k == "crash" or (isinstance(v, str) and v.startswith("ERROR"))]
_bad += [k for k, v in TOOL.items() if k != "files" and v.get("rc") != 0]
_bad += [f"missing {k}" for k, ok in TOOL["files"].items() if not ok]
print("\nPROBE SUMMARY - not OK on this CPU machine (recorded, reviewed before the GPU notebook):", _bad or "none")

## 4. Launch-flag check

In [ ]:
# 4. The Experiment 11 launch flags must exist in this build (help text of sglang.launch_server)
FLAGS = ["--model-path", "--tp-size", "--quantization", "--kv-cache-dtype", "--attention-backend", "--mem-fraction-static",
         "--max-running-requests", "--chunked-prefill-size", "--max-prefill-tokens", "--context-length",
         "--mamba-radix-cache-strategy", "--mamba-ssm-dtype", "--mamba-full-memory-ratio", "--speculative-algorithm",
         "--speculative-draft-model-path", "--speculative-draft-model-quantization", "--speculative-num-draft-tokens",
         "--speculative-num-steps", "--speculative-eagle-topk", "--reasoning-parser", "--enable-metrics",
         "--disable-flashinfer-autotune"]
r = subprocess.run([str(VPY), "-m", "sglang.launch_server", "--help"], env=env, capture_output=True, text=True, timeout=1800)
helptext = r.stdout + r.stderr
missing = [f for f in FLAGS if f not in helptext]
print("usage printed:", "usage" in helptext.lower(), "| flags not listed:", missing or "none")
if "usage" not in helptext.lower():
    print(helptext[-3000:])
(WH / "BUILD_FLAGS.json").write_text(json.dumps({"usage_printed": "usage" in helptext.lower(), "missing": missing}, indent=2),
                                     encoding="utf-8")

## 5. Checksums, README

In [ ]:
# 5. Checksums, build info, README (then free the test install)
info = {
    "sglang": SGLANG_VERSION, "python": sys.version.split()[0],
    "glibc": subprocess.run(["ldd", "--version"], capture_output=True, text=True).stdout.splitlines()[0],
    "built_utc": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z", "built_on": "Kaggle",
    "wheel_count": len(list(WH.glob("*.whl"))),
    "note": "cuda-tile overridden 1.6.0rc5 -> 1.6.0 (rc5 has no wheel); nvcc/crt/nvvm/nvjitlink pinned 13.0.88 (torch "
            "2.13 cuda-toolkit 13.0.3 set, Kaggle L4 driver = CUDA 13.0); install with install_wheelhouse.py (venv, "
            "--no-deps, restores the '+' Kaggle drops from file names)",
    "dataset_version_notes": VERSION_NOTES,
    "stamp": f"sglang=={SGLANG_VERSION} torch=={PINS['torch']} flashinfer=={PINS['flashinfer-python']} "
             f"jit-cache=={PINS['flashinfer-jit-cache']} transformers=={PINS['transformers']}",
    "probe": PROBE_RESULT, "pins": PINS,
}
(WH / "BUILD_INFO.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
(WH / "README.md").write_text(
    f"# SGLang {SGLANG_VERSION} offline wheelhouse (built on Kaggle, CPython {info['python']}, CUDA 13 wheels)\n\n"
    "Install (one command, offline, CPython 3.12 Linux x86_64):\n\n"
    "    python /kaggle/input/<this dataset>/install_wheelhouse.py /tmp/sglang-venv\n\n"
    "Do NOT run `pip install --find-links <this dir>` directly: Kaggle drops '+' from file names, so\n"
    "flashinfer_jit_cache-0.6.18+cu130-...whl is stored as ...-0.6.18cu130-...whl and pip cannot match the lock.\n"
    "install_wheelhouse.py restores the names from SHA256SUMS (checking their sha256), creates the venv, installs\n"
    "requirements.lock with --no-deps and adds lib/libcudart.so + lib64 under the pip CUDA runtime. It prints\n"
    "VENV_PYTHON, CUDA_HOME and PATH: start SGLang with them and fresh SGLANG_CACHE_DIR / CUDA_CACHE_PATH.\n"
    "A venv (not --target + PYTHONPATH) so that .pth files run (nvidia-cutlass-dsl) and console scripts (ninja) are\n"
    "on PATH. On Kaggle also symlink libcuda.so.1 as libcuda.so into a LIBRARY_PATH directory (FlashInfer JIT links -lcuda).\n\n"
    "Made for ARC-AGI-2 Experiment 11 (RedHat Qwen3.8-27B INT4 + DFlash2 on 4x L4). The lock overrides cuda-tile\n"
    "1.6.0rc5 -> 1.6.0 because rc5 is sdist-only; --no-deps keeps pip from re-checking that pin. nvcc/crt/nvvm/nvjitlink\n"
    "are pinned to 13.0.88 (torch 2.13's cuda-toolkit 13.0.3 set; the Kaggle L4 driver supports CUDA 13.0).\n"
    "BUILD_PROBE.json (imports + CPU nvcc toolchain) / BUILD_FLAGS.json record the CPU-side checks done at build time.\n",
    encoding="utf-8")
files = sorted(p for p in WH.iterdir() if p.is_file() and p.name != "SHA256SUMS")
(WH / "SHA256SUMS").write_text("".join(f"{WHEEL_SUMS.get(p.name) or sha256(p)}  {p.name}\n" for p in files),
                               encoding="utf-8")
for _d in (VENV, Path(str(VENV) + "-wheels"), SIM):
    shutil.rmtree(_d, ignore_errors=True)
print(info["stamp"])
print("upload size:", round(sum(p.stat().st_size for p in WH.iterdir()) / 1024**3, 2), "GiB")

## 6. Publish the dataset

In [ ]:
# 6. Publish as a dataset (Kaggle -> Kaggle, uses the attached Secrets)
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
KAGGLE_USERNAME = s.get_secret("KAGGLE_USERNAME").strip()
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = s.get_secret("KAGGLE_KEY").strip()
cred = Path.home() / ".kaggle" / "kaggle.json"
cred.parent.mkdir(parents=True, exist_ok=True)
cred.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": os.environ["KAGGLE_KEY"]}))
cred.chmod(0o600)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.4"], check=True)

meta = {"title": DATASET_TITLE, "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}", "licenses": [{"name": "other"}]}
(WH / "dataset-metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(json.dumps(meta, indent=1))
REF = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
_st = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "status", REF], capture_output=True, text=True)
EXISTS = _st.returncode == 0 and "ready" in (_st.stdout + _st.stderr).lower()
if EXISTS:      # v2: run 1 created the dataset -> publish a new version of it
    cmd = [sys.executable, "-m", "kaggle", "datasets", "version", "-p", str(WH), "-m", VERSION_NOTES, "-r", "skip"]
else:
    cmd = [sys.executable, "-m", "kaggle", "datasets", "create", "-p", str(WH), "-r", "skip"] + (["--public"] if PUBLIC else [])
print("dataset exists:", EXISTS, "->", " ".join(cmd[3:5]))
r = subprocess.run(cmd, text=True)
print("exit code:", r.returncode)
# Run 1: the create call printed "Expecting value" (a non-JSON reply) and exited 1, yet the dataset appeared a little later.
# So the exit code alone is not trusted: wait until the new file install_wheelhouse.py is listed in the dataset.
t0 = time.time()
listed = False
while time.time() - t0 < 25 * 60:
    _ls = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "files", REF, "--page-size", "300"],
                         capture_output=True, text=True)
    if "install_wheelhouse.py" in _ls.stdout:
        listed = True
        break
    time.sleep(30)
print(f"new version listed: {listed} after {time.time() - t0:.0f}s")
subprocess.run([sys.executable, "-m", "kaggle", "datasets", "status", REF])
if not listed:
    raise SystemExit("the new dataset version is not visible after 25 min - paste this output back")
print("attach:", REF, "| mount: /kaggle/input/datasets/" + REF)